In [17]:
import os
import glob, re
import pandas as pd
import shutil
from tqdm import tqdm
from pathlib import Path
import shutil

In [ ]:
# Pandas :: view ALL
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Pandas :: Default view
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

In [51]:
base   = r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028"
RT_dir = os.path.join(base, "8641_Dem+MCI+AD")
USED_dir = os.path.join(RT_dir, "USED")
files = r"D:\JHAhn\files"

# 1
xlsx_in = r"D:\JHAhn\files\안재혁_8641_Auto_ALPS_index_19-24_main+CTI.xlsx"
USED = os.path.join(RT_dir, "USED")

# 2

# 3


In [3]:
# Prepare 'tqdm'
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, *a, **k): return x

C:\Users\user\anaconda3\envs\JHA_dev\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Prepare USED individuals as each Folders in :

In [8]:
OVERWRITE_IF_EXISTS = False
DRY_RUN = False

df = pd.read_excel(xlsx_in, usecols=["PDI"])
pdis = (
    df["PDI"]
    .astype(str)
    .str.strip()
    .replace({"": pd.NA})
    .dropna()
    .drop_duplicates()
    .tolist()
)
df

,PDI
0,ADEPT54_HCS_NODDI24
1,EPT2022AD01_KDKM_NODDI32
2,EPT2022AD02_SKM_NODDI32
3,EPT2022AD03_KKS_NODDI32
4,EPT2022AD05_KIJD_NODDI32
...,...
97,MWI_CN79_CYE_NODDI32
98,MWI_CN80_LSR_NODDI32
99,MWI_CN81_CSH_NODDI32
100,MWI_CN82_LMR_NODDI32


In [9]:
os.makedirs(USED, exist_ok=True)

In [10]:
moved, skipped_exists, not_found, errors = [], [], [], []

for name in tqdm(pdis, desc="Moving selected subjects", unit="person"):
    src = os.path.join(RT_dir, name)
    if not os.path.isdir(src):
        not_found.append(name)
        continue

    dst = os.path.join(USED, name)
    if os.path.exists(dst):
        if OVERWRITE_IF_EXISTS:
            # 기존 대상 삭제 후 이동 (주의!)
            try:
                if not DRY_RUN:
                    shutil.rmtree(dst)
            except Exception as e:
                errors.append((name, f"remove dst failed: {e}"))
                continue
        else:
            skipped_exists.append(name)
            continue

    try:
        if not DRY_RUN:
            shutil.move(src, dst)
        moved.append(name)
    except Exception as e:
        errors.append((name, str(e)))

# Printing
total = len(pdis)
matched_in_source = len(moved) + len(skipped_exists)  # 소스에 폴더 존재했던 수
moved_ratio = f"{len(moved)}/{total} ({(len(moved)/total*100):.1f}%)"
matched_ratio = f"{matched_in_source}/{total} ({(matched_in_source/total*100):.1f}%)"

print("\n=== SUMMARY ===")
print(f"Total in Excel (unique PDI): {total}")
print(f"Matched in source: {matched_ratio}")          # 소스에 폴더 있었던 수(이동+이미 USED)
print(f"Matched & moved this run: {moved_ratio}")     # 이번 실행에서 실제 이동된 수
print(f"Already in USED (skipped): {len(skipped_exists)}")
print(f"Not found in source: {len(not_found)}")
print(f"Errors: {len(errors)}")
if DRY_RUN:
    print("\n[DRY RUN] !!!! Movement NOT EXECUTED !!!!")

# --- 로그 저장 (base에 저장) ---
log_path = os.path.join(base, "USED_move_log.xlsx")
with pd.ExcelWriter(log_path, engine="xlsxwriter") as xw:
    pd.DataFrame({"PDI": moved}).to_excel(xw, index=False, sheet_name="moved")
    pd.DataFrame({"PDI": skipped_exists}).to_excel(xw, index=False, sheet_name="skipped_exists")
    pd.DataFrame({"PDI": not_found}).to_excel(xw, index=False, sheet_name="not_found")
    if errors:
        pd.DataFrame(errors, columns=["PDI", "error"]).to_excel(xw, index=False, sheet_name="errors")

print(f"\n[OK] Log saved: {log_path}")
print(f"[INFO] Files moved to: {USED}")

Moving selected subjects: 100%|█████████████████████████████████████████████████| 102/102 [00:00<00:00, 144.22person/s]


=== SUMMARY ===
Total in Excel (unique PDI): 102
Matched in source: 101/102 (99.0%)
Matched & moved this run: 101/102 (99.0%)
Already in USED (skipped): 0
Not found in source: 1
Errors: 0

[OK] Log saved: D:\JHAhn\8641_conductivity\DTI_ALPS\1028\USED_move_log.xlsx
[INFO] Files moved to: D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD\USED


In [11]:
unmatched = sorted(set(not_found))  # 중복 제거 + 정렬
print(f"[UNMATCHED] {len(unmatched)}명")
for name in unmatched:
    print(name)

[UNMATCHED] 1명
HFC2021AD29_LBD_NODDI32


In [ ]:
## Conductivity data merging

In [52]:
ROI_DIR    = r"C:\Users\user\Documents\MATLAB\wfu_results\ROI_txt"

PDI_ORDER = [
    "EPT2022AD06_JOJ_NODDI32",
    "EPT2022AD35_HHS_NODDI32",
    "EPT2024AD09_CSO_NODDI32",
    "MREPT2020AD01_CGS_NODDI32",
    "MREPT2020AD07_KYG_NODDI32",
    "MREPT2020AD18_YHS_NODDI32",
    "MREPT2020AD24_YYS_NODDI32",
    "MREPT2020AD28_YYL_NODDI32",
    "MWI_CN17_LPS_NODDI32",
    "MWI_CN18_HCS_NODDI32",
]

# txt 파일 수집
txt_files = sorted(Path(ROI_DIR).glob("*.txt"))

# 병합
data = {"PDI": PDI_ORDER}
n = len(PDI_ORDER)

for fp in txt_files:
    with open(fp, "r", errors="ignore") as f:
        vals = [float(line.strip()) for line in f if line.strip()]
    if len(vals) != n:
        raise ValueError(f"[길이불일치] {fp.name}: {len(vals)}행 != PDI {n}명")
    data[fp.stem] = vals

df = pd.DataFrame(data)

# Save
save_path = os.path.join(files, "conductivities_ADD-individuals.xlsx")
df.to_excel(save_path, index=False)

print("[OK]", save_path)

df

[OK] D:\JHAhn\files\conductivities_ADD-individuals.xlsx


,PDI,EC_LT_Amygdala,EC_LT_Hippocampus,EC_LT_Insula,EC_LT_MTG,EC_LT_ParaHippocampus,EC_LT_Thalamus,EC_RT_Amygdala,EC_RT_Hippocampus,EC_RT_Insula,...,IC_LT_ParaHippocampus,IC_LT_Thalamus,IC_RT_Amygdala,IC_RT_Hippocampus,IC_RT_Insula,IC_RT_MTG,IC_RT_ParaHippocampus,IC_RT_Thalamus,IC_Whole_CC,IC_Whole_Precuneus
0,EPT2022AD06_JOJ_NODDI32,0.694346,0.675511,0.518085,0.456421,0.799331,0.510635,0.486143,0.570023,0.580223,...,0.099184,0.169846,0.074682,0.081098,0.070232,0.081215,0.096972,0.171253,0.148262,0.083440
1,EPT2022AD35_HHS_NODDI32,0.495155,0.638647,0.462557,0.442096,0.503966,0.480025,0.399267,0.671910,0.611697,...,0.089590,0.283925,0.081331,0.058594,0.105213,0.098891,0.074542,0.284467,0.112889,0.137250
2,EPT2024AD09_CSO_NODDI32,0.492203,0.627809,0.493964,0.473858,0.659371,0.499633,0.561331,0.640562,0.560045,...,0.098535,0.217729,0.105398,0.095026,0.086558,0.104253,0.102353,0.242145,0.143183,0.122816
3,MREPT2020AD01_CGS_NODDI32,0.646418,0.828780,0.495569,0.518030,0.881444,0.682093,0.577818,0.695979,0.569704,...,0.161364,0.171456,0.085974,0.085251,0.092273,0.108377,0.133684,0.196702,0.110468,0.118045
4,MREPT2020AD07_KYG_NODDI32,0.546536,0.668408,0.566353,0.592762,0.664199,0.602866,0.505585,0.680955,0.625673,...,0.088440,0.235402,0.134927,0.079246,0.103106,0.106203,0.091974,0.248849,0.121573,0.103760
5,MREPT2020AD18_YHS_NODDI32,0.679963,0.701162,0.552641,0.491568,0.654551,0.721738,0.581802,0.581239,0.611053,...,0.101314,0.228767,0.129536,0.092743,0.087380,0.103569,0.089409,0.250838,0.205110,0.130868
6,MREPT2020AD24_YYS_NODDI32,0.472527,0.593464,0.492699,0.498880,0.633285,0.598319,0.556453,0.807122,0.718111,...,0.084657,0.210674,0.130427,0.126549,0.067864,0.110994,0.146734,0.325800,0.143295,0.172827
7,MREPT2020AD28_YYL_NODDI32,0.671291,0.704186,0.539392,0.626136,0.782249,0.620370,0.837254,0.679370,0.713869,...,0.127544,0.152677,0.185611,0.069797,0.087423,0.106724,0.118320,0.230867,0.131347,0.130565
8,MWI_CN17_LPS_NODDI32,0.496523,0.501701,0.412381,0.408535,0.583225,0.610159,0.440644,0.498299,0.486381,...,0.132417,0.231018,0.089912,0.112144,0.093643,0.111033,0.111464,0.244920,0.176074,0.153292
9,MWI_CN18_HCS_NODDI32,0.416537,0.449563,0.430492,0.422270,0.595918,0.497247,0.482068,0.508369,0.548816,...,0.105976,0.244833,0.128036,0.127172,0.078967,0.106848,0.123208,0.303949,0.191792,0.134528


In [ ]:
## CTI/DTI ALPS Results to dF

In [23]:
COLS = [
    "PDI",
    "LT_ALL_Dxx_proj","LT_ALL_Dyy_proj","LT_ALL_Dzz_proj","LT_ALL_Dxx_assoc","LT_ALL_Dyy_assoc","LT_ALL_Dzz_assoc","LT_ALL_ALPS index",
    "LT_b800_Dxx_proj","LT_b800_Dyy_proj","LT_b800_Dzz_proj","LT_b800_Dxx_assoc","LT_b800_Dyy_assoc","LT_b800_Dzz_assoc","LT_b800_ALPS index",
    "LT_b2000_Dxx_proj","LT_b2000_Dyy_proj","LT_b2000_Dzz_proj","LT_b2000_Dxx_assoc","LT_b2000_Dyy_assoc","LT_b2000_Dzz_assoc","LT_b2000_ALPS index",
    "RT_ALL_Dxx_proj","RT_ALL_Dyy_proj","RT_ALL_Dzz_proj","RT_ALL_Dxx_assoc","RT_ALL_Dyy_assoc","RT_ALL_Dzz_assoc","RT_ALL_ALPS index",
    "RT_b800_Dxx_proj","RT_b800_Dyy_proj","RT_b800_Dzz_proj","RT_b800_Dxx_assoc","RT_b800_Dyy_assoc","RT_b800_Dzz_assoc","RT_b800_ALPS index",
    "RT_b2000_Dxx_proj","RT_b2000_Dyy_proj","RT_b2000_Dzz_proj","RT_b2000_Dxx_assoc","RT_b2000_Dyy_assoc","RT_b2000_Dzz_assoc","RT_b2000_ALPS index",
    "LT_CTI_Proj_Dxx","LT_CTI_Proj_Dyy","LT_CTI_Proj_Dzz","LT_CTI_Assoc_Dxx","LT_CTI_Assoc_Dyy","LT_CTI_Assoc_Dzz","LT_CTI-ALPS index",
    "RT_CTI_Proj_Dxx","RT_CTI_Proj_Dyy","RT_CTI_Proj_Dzz","RT_CTI_Assoc_Dxx","RT_CTI_Assoc_Dyy","RT_CTI_Assoc_Dzz","RT_CTI-ALPS index",
]

MAP = {
    "Auto_ALPS_index_LH_MNIspace.csv": [
        "LT_ALL_Dxx_proj","LT_ALL_Dyy_proj","LT_ALL_Dzz_proj","LT_ALL_Dxx_assoc","LT_ALL_Dyy_assoc","LT_ALL_Dzz_assoc","LT_ALL_ALPS index"
    ],
    "Auto_ALPS_index_LH_maxbval800_MNIspace.csv": [
        "LT_b800_Dxx_proj","LT_b800_Dyy_proj","LT_b800_Dzz_proj","LT_b800_Dxx_assoc","LT_b800_Dyy_assoc","LT_b800_Dzz_assoc","LT_b800_ALPS index"
    ],
    "Auto_ALPS_index_LH_maxbval2000_MNIspace.csv": [
        "LT_b2000_Dxx_proj","LT_b2000_Dyy_proj","LT_b2000_Dzz_proj","LT_b2000_Dxx_assoc","LT_b2000_Dyy_assoc","LT_b2000_Dzz_assoc","LT_b2000_ALPS index"
    ],
    "Auto_ALPS_index_RH_MNIspace.csv": [
        "RT_ALL_Dxx_proj","RT_ALL_Dyy_proj","RT_ALL_Dzz_proj","RT_ALL_Dxx_assoc","RT_ALL_Dyy_assoc","RT_ALL_Dzz_assoc","RT_ALL_ALPS index"
    ],
    "Auto_ALPS_index_RH_maxbval800_MNIspace.csv": [
        "RT_b800_Dxx_proj","RT_b800_Dyy_proj","RT_b800_Dzz_proj","RT_b800_Dxx_assoc","RT_b800_Dyy_assoc","RT_b800_Dzz_assoc","RT_b800_ALPS index"
    ],
    "Auto_ALPS_index_RH_maxbval2000_MNIspace.csv": [
        "RT_b2000_Dxx_proj","RT_b2000_Dyy_proj","RT_b2000_Dzz_proj","RT_b2000_Dxx_assoc","RT_b2000_Dyy_assoc","RT_b2000_Dzz_assoc","RT_b2000_ALPS index"
    ],
    "Auto_CTI_ALPS_index_LH.csv": [
        "LT_CTI_Proj_Dxx","LT_CTI_Proj_Dyy","LT_CTI_Proj_Dzz","LT_CTI_Assoc_Dxx","LT_CTI_Assoc_Dyy","LT_CTI_Assoc_Dzz","LT_CTI-ALPS index"
    ],
    "Auto_CTI_ALPS_index_RH.csv": [
        "RT_CTI_Proj_Dxx","RT_CTI_Proj_Dyy","RT_CTI_Proj_Dzz","RT_CTI_Assoc_Dxx","RT_CTI_Assoc_Dyy","RT_CTI_Assoc_Dzz","RT_CTI-ALPS index"
    ],
}

In [19]:
'''
def parse_dti_csv(fp: str):
    try:
        df = pd.read_csv(fp)
        df.columns = [c.strip() for c in df.columns]
        keys = ["Dxx_proj","Dyy_proj","Dzz_proj","Dxx_assoc","Dyy_assoc","Dzz_assoc","ALPS index"]
        if all(k in df.columns for k in keys):
            row = df.iloc[0][keys]
            return [pd.to_numeric(x, errors="coerce") for x in row.tolist()]
    except Exception:
        pass
    # 폴백: 텍스트에서 숫자 7개 추출
    with open(fp, "r", errors="ignore") as f:
        txt = f.read()
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", txt)
    return [float(x) for x in nums[:7]] if len(nums) >= 7 else [float("nan")]*7

def parse_cti_csv(fp: str):
    with open(fp, "r", errors="ignore") as f:
        txt = f.read()
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", txt)
    return [float(x) for x in nums[:7]] if len(nums) >= 7 else [float("nan")]*7

def parse_file(fp: str, fname: str):
    return parse_cti_csv(fp) if "CTI" in fname else parse_dti_csv(fp)

# Collect individual folders 
subjects = []
if os.path.isdir(USED):
    group_dirs = [os.path.join(USED, g) for g in os.listdir(USED) if os.path.isdir(os.path.join(USED, g))]
    for g in group_dirs:
        for d in os.listdir(g):
            p = os.path.join(g, d)
            if os.path.isdir(p):
                subjects.append(p)
else:
    # (옵션) USED가 없으면 이전 방식으로 RT_dir 바로 아래 폴더를 개인으로 간주
    subjects = [os.path.join(RT_dir, d) for d in os.listdir(RT_dir) if os.path.isdir(os.path.join(RT_dir, d))]

subjects.sort(key=lambda p: os.path.basename(p))

rows = []
for subj in tqdm(subjects, desc="Subjects", unit="person"):
    row = {c: pd.NA for c in COLS}
    row["PDI"] = os.path.basename(subj)
    stats_dir = os.path.join(subj, "Results", "Stats")
    if os.path.isdir(stats_dir):
        for fname, target_cols in MAP.items():
            fp = os.path.join(stats_dir, fname)
            if not os.path.exists(fp):
                continue
            vals = parse_file(fp, fname)
            for c, v in zip(target_cols, vals):
                row[c] = v
    rows.append(row)

df = pd.DataFrame(rows)[COLS]
print(f"[INFO] subjects found: {len(subjects)}")
print(f"[INFO] df shape: {df.shape}")
df
'''

Subjects: 100%|██████████████████████████████████████████████████████████████████| 110/110 [00:03<00:00, 34.38person/s]

[INFO] subjects found: 110
[INFO] df shape: (110, 57)


,PDI,LT_ALL_Dxx_proj,LT_ALL_Dyy_proj,LT_ALL_Dzz_proj,LT_ALL_Dxx_assoc,LT_ALL_Dyy_assoc,LT_ALL_Dzz_assoc,LT_ALL_ALPS index,LT_b800_Dxx_proj,LT_b800_Dyy_proj,...,LT_CTI_Assoc_Dyy,LT_CTI_Assoc_Dzz,LT_CTI-ALPS index,RT_CTI_Proj_Dxx,RT_CTI_Proj_Dyy,RT_CTI_Proj_Dzz,RT_CTI_Assoc_Dxx,RT_CTI_Assoc_Dyy,RT_CTI_Assoc_Dzz,RT_CTI-ALPS index
0,EPT2022AD01_KDKM_NODDI32,0.698943,0.550937,1.132250,0.837376,1.101387,0.579667,1.358848,0.694262,0.550745,...,0.022996,0.121065,0.147115,2022.0,1.0,32.000000,0.054003,0.054375,0.240860,0.184253
1,EPT2022AD02_SKM_NODDI32,0.908146,0.925644,1.395797,0.957882,1.052785,0.782825,1.092222,0.904188,0.926162,...,0.469037,0.678170,0.611230,2022.0,2.0,32.000000,0.571705,0.575767,0.810732,0.635722
2,EPT2022AD03_KKS_NODDI32,0.675541,0.582997,1.289094,0.900925,1.152915,0.669345,1.258814,0.670758,0.579454,...,0.055372,0.255793,0.172951,2022.0,3.0,32.000000,0.162839,0.164997,0.514460,0.213250
3,EPT2022AD05_KIJD_NODDI32,0.689047,0.554143,0.990909,0.877817,0.967549,0.616083,1.338941,0.686075,0.550716,...,0.207067,0.459907,0.319917,2022.0,5.0,32.000000,0.320157,0.311376,0.667341,0.353844
4,EPT2022AD063_BPB_NODDI32,0.738484,0.675358,1.112329,0.828876,1.039489,0.649739,1.182827,0.740547,0.670570,...,0.058749,0.192047,0.178539,2022.0,63.0,32.000000,0.070293,0.067808,0.255883,0.215472
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,MWI_CN79_CYE_NODDI32,0.766420,0.540899,1.173629,0.870478,1.079347,0.589004,1.448707,0.764644,0.537796,...,0.132433,0.206054,0.208444,79.0,32.0,0.049923,0.049991,0.204732,0.167182,0.183320
106,MWI_CN80_LSR_NODDI32,0.628933,0.570822,1.051915,0.861719,1.005305,0.652572,1.218456,0.628976,0.567879,...,0.335069,0.273815,0.298218,80.0,32.0,0.106818,0.108289,0.381302,0.167943,0.310890
107,MWI_CN81_CSH_NODDI32,0.655519,0.606193,1.117180,0.774778,0.984770,0.566668,1.219494,0.651062,0.603767,...,0.278866,0.228572,0.261601,81.0,32.0,0.125594,0.115952,0.334596,0.239380,0.346851
108,MWI_CN82_LMR_NODDI32,0.800717,0.698031,1.304652,0.863600,1.115128,0.594507,1.287635,0.797227,0.697845,...,0.324074,0.201112,0.291503,82.0,32.0,0.212786,0.215661,0.490228,0.226204,0.382458


In [53]:
def parse_dti_csv(fp: str):
    try:
        df = pd.read_csv(fp)
        df.columns = [c.strip() for c in df.columns]
        keys = ["Dxx_proj","Dyy_proj","Dzz_proj","Dxx_assoc","Dyy_assoc","Dzz_assoc","ALPS index"]
        if all(k in df.columns for k in keys):
            row = df.iloc[0][keys]
            return [pd.to_numeric(x, errors="coerce") for x in row.tolist()]
    except Exception:
        pass
    with open(fp, "r", errors="ignore") as f:
        txt = f.read()
    nums = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', txt)
    return [float(x) for x in nums[:7]] if len(nums) >= 7 else [float("nan")]*7

def _to_float(tok: str):
    if tok is None: return float("nan")
    s = str(tok).strip().strip('"').strip("'")
    m = re.search(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', s)
    return float(m.group(0)) if m else float("nan")

def parse_cti_csv(fp: str):
    # 첫 비어있지 않은 라인 → 첫 토큰(PID) 무시, 이후 7개 숫자
    first = None
    with open(fp, "r", errors="ignore") as f:
        for line in f:
            if line.strip():
                first = line
                break
    if first is None:
        return [float("nan")]*7

    toks = first.strip().split(",") if "," in first else first.strip().split()
    vals = [_to_float(t) for t in toks[1:8]]  # PID 건너뛰고 7개

    # 부족/NaN 폴백: pandas로 재시도
    if len(vals) < 7 or any(pd.isna(x) for x in vals):
        try:
            df_try = pd.read_csv(fp, header=None, engine="python")
            if not df_try.empty:
                row0 = df_try.iloc[0].tolist()
                vals = [_to_float(t) for t in row0[1:8]]
        except Exception:
            pass

    # 최종 보정
    if len(vals) < 7: vals += [float("nan")]*(7-len(vals))
    return vals

def parse_file(fp: str, fname: str):
    return parse_cti_csv(fp) if "CTI" in fname.upper() else parse_dti_csv(fp)

# === 개인 폴더 수집: USED/<group>/<person> ===
subjects = []
if os.path.isdir(USED_dir):
    for g in os.listdir(USED_dir):
        gpath = os.path.join(USED_dir, g)
        if not os.path.isdir(gpath): continue
        for d in os.listdir(gpath):
            p = os.path.join(gpath, d)
            if os.path.isdir(p):
                subjects.append(p)
else:
    # 백업: USED가 없으면 RT_root 바로 아래 폴더를 개인으로 간주
    subjects = [os.path.join(RT_root, d) for d in os.listdir(RT_root) if os.path.isdir(os.path.join(RT_root, d))]

subjects.sort(key=lambda p: os.path.basename(p))

# === 메인 루프 ===
rows = []
for subj in tqdm(subjects, desc="Subjects", unit="person"):
    row = {c: pd.NA for c in COLS}
    row["PDI"] = os.path.basename(subj)
    stats_dir = os.path.join(subj, "Results", "Stats")
    if os.path.isdir(stats_dir):
        for fname, target_cols in MAP.items():
            fp = os.path.join(stats_dir, fname)
            if not os.path.exists(fp): 
                continue
            vals = parse_file(fp, fname)
            for c, v in zip(target_cols, vals):
                row[c] = v
    rows.append(row)

df_alps = pd.DataFrame(rows)[COLS]
print(f"[INFO] subjects found: {len(subjects)}")
print(f"[INFO] df shape: {df.shape}")
df_alps

Subjects: 100%|██████████████████████████████████████████████████████████████████| 112/112 [00:01<00:00, 75.36person/s]

[INFO] subjects found: 112
[INFO] df shape: (10, 43)


,PDI,LT_ALL_Dxx_proj,LT_ALL_Dyy_proj,LT_ALL_Dzz_proj,LT_ALL_Dxx_assoc,LT_ALL_Dyy_assoc,LT_ALL_Dzz_assoc,LT_ALL_ALPS index,LT_b800_Dxx_proj,LT_b800_Dyy_proj,...,LT_CTI_Assoc_Dyy,LT_CTI_Assoc_Dzz,LT_CTI-ALPS index,RT_CTI_Proj_Dxx,RT_CTI_Proj_Dyy,RT_CTI_Proj_Dzz,RT_CTI_Assoc_Dxx,RT_CTI_Assoc_Dyy,RT_CTI_Assoc_Dzz,RT_CTI-ALPS index
0,ADD,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,EPT2022AD01_KDKM_NODDI32,0.698943,0.550937,1.13225,0.837376,1.101387,0.579667,1.358848,0.694262,0.550745,...,0.164354,0.09731,1.47838,0.054003,0.054375,0.24086,0.184253,0.287577,0.178141,1.024686
2,EPT2022AD02_SKM_NODDI32,0.908146,0.925644,1.395797,0.957882,1.052785,0.782825,1.092222,0.904188,0.926162,...,0.662543,0.608131,1.031772,0.571705,0.575767,0.810732,0.635722,0.745992,0.640786,0.992498
3,EPT2022AD03_KKS_NODDI32,0.675541,0.582997,1.289094,0.900925,1.152915,0.669345,1.258814,0.670758,0.579454,...,0.218842,0.130708,1.237209,0.162839,0.164997,0.51446,0.21325,0.368805,0.217572,0.983061
4,EPT2022AD05_KIJD_NODDI32,0.689047,0.554143,0.990909,0.877817,0.967549,0.616083,1.338941,0.686075,0.550716,...,0.302673,0.241407,1.154961,0.320157,0.311376,0.667341,0.353844,0.461456,0.331211,1.048886
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107,MWI_CN80_LSR_NODDI32,0.628933,0.570822,1.051915,0.861719,1.005305,0.652572,1.218456,0.628976,0.567879,...,0.298218,0.207423,1.221575,0.106818,0.108289,0.381302,0.167943,0.31089,0.185265,0.935977
108,MWI_CN81_CSH_NODDI32,0.655519,0.606193,1.11718,0.774778,0.98477,0.566668,1.219494,0.651062,0.603767,...,0.261601,0.194056,1.164622,0.125594,0.115952,0.334596,0.23938,0.346851,0.263889,0.960862
109,MWI_CN82_LMR_NODDI32,0.800717,0.698031,1.304652,0.8636,1.115128,0.594507,1.287635,0.797227,0.697845,...,0.291503,0.176816,1.164496,0.212786,0.215661,0.490228,0.226204,0.382458,0.211427,1.027867
110,MWI_CN83_LYS_NODDI32,0.577291,0.513866,1.078292,0.741111,0.887641,0.497596,1.303462,0.573492,0.5086,...,0.27475,0.152755,1.206411,0.120062,0.120405,0.290888,0.163748,0.200106,0.144309,1.072138


In [26]:
## SAVE df

# XLSX
xlsx_path = os.path.join(base, "Auto_ALPS_DTI+CTI_1120.xlsx")
with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as xw:
    df_alps.to_excel(xw, index=False, sheet_name="merged")

print(f"[OK] Saved:\n - {xlsx_path}")

[OK] Saved:
 - D:\JHAhn\8641_conductivity\DTI_ALPS\1028\Auto_ALPS_DTI+CTI_1120.xlsx


In [ ]:
# Tissue Volumes (TiV) to dF

In [54]:
group_dirs = [os.path.join(USED, g) for g in os.listdir(USED) if os.path.isdir(os.path.join(USED, g))]
subjects = []
for g in group_dirs:
    for d in os.listdir(g):
        p = os.path.join(g, d)
        if os.path.isdir(p):
            subjects.append(p)

# Sort(by folder names of each individual)
subjects.sort(key=lambda p: os.path.basename(p))

rows = []
cols_template = None
found, not_found = [], []

for subj in tqdm(subjects, desc="Collect BrainVolumeStats (USED/<group>/<person>)", unit="person"):
    pid_folder = os.path.basename(subj)
    fpath = os.path.join(subj, "Results", "Stats", "eTIV_NormalizedBrainVolumeStats.csv")
    if not os.path.exists(fpath):
        not_found.append(os.path.join(os.path.basename(os.path.dirname(subj)), pid_folder))  # group/person 표기
        continue

    df = pd.read_csv(fpath, sep=None, engine="python")
    df.columns = [str(c).strip() for c in df.columns]
    if "Measure:volume" not in df.columns:
        raise ValueError(f"[{pid_folder}] 'Measure:volume' 컬럼을 찾지 못했습니다: {fpath}")

    if cols_template is None:
        cols_template = [c for c in df.columns if c != "Measure:volume"]

    row = df.iloc[0].copy()
    pdi = str(row["Measure:volume"]).strip()
    row = row.drop(labels=["Measure:volume"]).apply(pd.to_numeric, errors="coerce")

    out = {"PDI": pdi}
    for c in cols_template:
        out[c] = row.get(c, pd.NA)
    rows.append(out)
    found.append(pid_folder)

# Merge-Summary-Save
df_tivs = pd.DataFrame(rows, columns=["PDI"] + (cols_template or []))

print("\n=== SUMMARY ===")
print(f"Groups under USED: {len(group_dirs)}")
print(f"Persons discovered: {len(subjects)}")
print(f"Found BrainVolumeStats.csv: {len(found)}")
print(f"Merged rows: {len(merged)}")
print(f"Missing BrainVolumeStats.csv: {len(not_found)}")

if not_found:
    print("\n[UNMATCHED - no BrainVolumeStats.csv] (group/person)")
    for n in sorted(not_found):
        print(" -", n)

df_tivs

Collect BrainVolumeStats (USED/<group>/<person>): 100%|█████████████████████████| 112/112 [00:00<00:00, 174.01person/s]


=== SUMMARY ===
Groups under USED: 4
Persons discovered: 112
Found BrainVolumeStats.csv: 110
Merged rows: 110
Missing BrainVolumeStats.csv: 2

[UNMATCHED - no BrainVolumeStats.csv] (group/person)
 - marsbar_conduct\ADD
 - marsbar_conduct\bin


,PDI,Left-Cerebral-White-Matter,Left-Lateral-Ventricle,Left-Inf-Lat-Vent,Left-Cerebellum-White-Matter,Left-Cerebellum-Cortex,Left-Thalamus,Left-Caudate,Left-Putamen,Left-Pallidum,...,ctx-rh-precuneus,ctx-rh-rostralanteriorcingulate,ctx-rh-rostralmiddlefrontal,ctx-rh-superiorfrontal,ctx-rh-superiorparietal,ctx-rh-superiortemporal,ctx-rh-supramarginal,ctx-rh-transversetemporal,ctx-rh-insula,EstimatedTotalIntraCranialVol
0,EPT2022AD01_KDKM_NODDI32,20.402607,1.306631,0.034820,1.298794,4.383638,0.604671,0.269637,0.373014,0.160117,...,0.716326,0.201805,0.930315,1.793907,0.851352,1.143155,0.785335,0.060005,0.467622,100.0
1,EPT2022AD02_SKM_NODDI32,13.948878,2.663107,0.118179,0.786921,2.706773,0.352440,0.353675,0.285620,0.119545,...,0.377658,0.128418,0.460491,1.123300,0.617562,0.718252,0.333364,0.031503,0.230763,100.0
2,EPT2022AD03_KKS_NODDI32,13.195277,2.364204,0.128392,0.759140,3.254569,0.353944,0.231111,0.249268,0.075292,...,0.459566,0.094796,0.598994,1.434607,0.540411,0.783835,0.487025,0.037389,0.278055,100.0
3,EPT2022AD05_KIJD_NODDI32,16.195046,1.585224,0.086319,1.056247,3.712048,0.450600,0.201113,0.253552,0.120831,...,0.569262,0.145015,0.667603,1.679516,0.694868,0.813188,0.538963,0.043187,0.362993,100.0
4,EPT2022AD063_BPB_NODDI32,16.704450,1.196887,0.041819,0.856983,3.282660,0.404151,0.240808,0.268180,0.107072,...,0.474612,0.109571,0.486127,1.315480,0.539062,0.703939,0.441195,0.043831,0.254882,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,MWI_CN79_CYE_NODDI32,20.069944,1.132137,0.062483,1.141053,4.481554,0.565964,0.341817,0.393089,0.169199,...,0.748116,0.146930,0.759841,2.094886,0.774119,1.210641,0.728752,0.054634,0.495852,100.0
106,MWI_CN80_LSR_NODDI32,13.823590,1.021002,0.036493,0.985769,3.651302,0.460118,0.203395,0.279527,0.107572,...,0.617318,0.153809,0.661331,1.562381,0.739511,1.042161,0.496283,0.053774,0.393696,100.0
107,MWI_CN81_CSH_NODDI32,20.661109,1.767533,0.044536,1.379365,4.387010,0.583606,0.238289,0.344173,0.152596,...,0.736789,0.175744,0.871631,1.887091,0.841582,1.153358,0.641169,0.071320,0.440365,100.0
108,MWI_CN82_LMR_NODDI32,17.239570,1.600418,0.039387,0.965257,3.242787,0.437256,0.188738,0.314780,0.142754,...,0.669777,0.147082,0.761443,1.636091,0.665946,0.860666,0.610475,0.056442,0.390934,100.0


In [29]:
# Save

xlsx_out = os.path.join(base, "BrainVolumeStats_merged_1120.xlsx")
with pd.ExcelWriter(xlsx_out, engine="xlsxwriter") as xw:
    df_tivs.to_excel(xw, index=False, sheet_name="merged")

print(f"\n[OK] Saved:\n - {xlsx_out}")


[OK] Saved:
 - D:\JHAhn\8641_conductivity\DTI_ALPS\1028\BrainVolumeStats_merged_1120.xlsx


In [ ]:
## collect Conductivity files

In [41]:
src_root = r"D:\JHAhn\8641_conductivity\nomalized_image\Conductivity\EPT_sigma_int"  # 4개 하위 폴더 포함
dst_root = r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD\USED\marsbar_conduct"
xlsx_in  = r"D:\JHAhn\files\finLIST-1120_8641_ALPS+TiV.xlsx"

In [42]:
OVERWRITE = False     # True면 대상 존재 시 덮어쓰기(폴더는 지우고 다시 복사)
DRY_RUN   = False     # True면 실제 복사하지 않고 계획만 출력

# PDI listing
df = pd.read_excel(xlsx_in, usecols=["PDI"])
pdis = (df["PDI"].astype(str).str.strip()
        .replace({"": pd.NA}).dropna().drop_duplicates().tolist())

# 소스의 1차 하위 폴더들(군 폴더 4개라고 했음)
group_dirs = [os.path.join(src_root, d) for d in os.listdir(src_root)
              if os.path.isdir(os.path.join(src_root, d))]
group_dirs.sort()

# Utils
def norm(s):  # 대소문자 무시, 공백 정리
    return re.sub(r"\s+", "", str(s)).lower()

norm_pdis = [norm(p) for p in pdis]
pdi_map   = dict(zip(norm_pdis, pdis))  # 정규화→원본 PDI

os.makedirs(dst_root, exist_ok=True)

moved_folders, moved_files = [], []
skipped_exists, not_found  = [], []
errors = []

# Matching, Copy
for gdir in tqdm(group_dirs, desc="Groups", unit="group"):
    entries = os.listdir(gdir)
    # 1) 폴더명 매칭(개인 폴더 전체 복사)
    for name in tqdm(entries, desc=os.path.basename(gdir), unit="item", leave=False):
        spath = os.path.join(gdir, name)
        base  = os.path.basename(spath)
        nbase = norm(base)

        # 어떤 PDI가 포함됐는지 확인(부분 문자열 매칭)
        hits = [p for p in norm_pdis if p in nbase]
        if not hits:
            continue

        # 매칭된 첫 PDI 기준으로 복사 경로 결정
        pdi_key = hits[0]
        pdi     = pdi_map[pdi_key]
        dst_pdi_dir = os.path.join(dst_root, pdi)
        os.makedirs(dst_pdi_dir, exist_ok=True)

        if os.path.isdir(spath):
            # 폴더 전체를 dst_root\{PDI}\{원폴더명} 으로 복사
            dst_path = os.path.join(dst_pdi_dir, base)
            if os.path.exists(dst_path):
                if OVERWRITE:
                    try:
                        if not DRY_RUN:
                            shutil.rmtree(dst_path)
                    except Exception as e:
                        errors.append((pdi, f"remove dst failed: {e}"))
                        continue
                else:
                    skipped_exists.append(dst_path)
                    continue
            try:
                if not DRY_RUN:
                    shutil.copytree(spath, dst_path)
                moved_folders.append((pdi, dst_path))
            except Exception as e:
                errors.append((pdi, f"copytree failed: {e}"))
        else:
            # 파일이면 dst_root\{PDI}\ 로 파일 복사
            dst_path = os.path.join(dst_pdi_dir, base)
            if os.path.exists(dst_path) and not OVERWRITE:
                skipped_exists.append(dst_path)
                continue
            try:
                if not DRY_RUN:
                    shutil.copy2(spath, dst_path)
                moved_files.append((pdi, dst_path))
            except Exception as e:
                errors.append((pdi, f"copy2 failed: {e}"))

# NOT Matched PDIs (이름이 파일/폴더에 안 들어간 경우)
matched_pdis = {p for p, _ in moved_folders} | {p for p, _ in moved_files} | \
               {pdi_map[p] for p in norm_pdis if any(p in norm(x) for gd in group_dirs for x in os.listdir(gd))}
unmatched = [p for p in pdis if p not in matched_pdis]

# Summarise
total = len(pdis)
copied = len({p for p, _ in moved_folders} | {p for p, _ in moved_files})
print("\n=== SUMMARY ===")
print(f"Groups scanned        : {len(group_dirs)}")
print(f"PDI in Excel          : {total}")
print(f"PDI matched & copied  : {copied}/{total} ({copied/total*100:.1f}%)")
print(f"Existing skipped      : {len(skipped_exists)}")
print(f"Unmatched (no hit)    : {len(unmatched)}")
print(f"Errors                : {len(errors)}")
if DRY_RUN:
    print("[DRY RUN] 실제 복사는 수행하지 않았습니다.")

# 값만 출력
if unmatched:
    print("\n[UNMATCHED]")
    for p in unmatched:
        print(p)

# Logs
try:
    log_xlsx = os.path.join(dst_root, "_copy_log.xlsx")
    with pd.ExcelWriter(log_xlsx, engine="xlsxwriter") as xw:
        pd.DataFrame(moved_folders, columns=["PDI","dst"]).to_excel(xw, index=False, sheet_name="moved_folders")
        pd.DataFrame(moved_files,   columns=["PDI","dst"]).to_excel(xw, index=False, sheet_name="moved_files")
        pd.DataFrame(skipped_exists, columns=["skipped_path"]).to_excel(xw, index=False, sheet_name="skipped_exists")
        pd.DataFrame(unmatched, columns=["PDI"]).to_excel(xw, index=False, sheet_name="unmatched")
        pd.DataFrame(errors, columns=["PDI","error"]).to_excel(xw, index=False, sheet_name="errors")
    print(f"\n[OK] Log saved: {log_xlsx}")
except Exception:
    pass

Groups: 100%|█████████████████████████████████████████████████████████████████████████| 4/4 [00:05<00:00,  1.35s/group]


=== SUMMARY ===
Groups scanned        : 4
PDI in Excel          : 110
PDI matched & copied  : 106/110 (96.4%)
Existing skipped      : 0
Unmatched (no hit)    : 4
Errors                : 0

[UNMATCHED]
MREPT2020AD02_HMJ_NODDI32
MREPT2020AD03_KJH_NODDI32
MREPT2020AD08_CYS_NODDI32
MWI_CN68_CDL_NODDI32

[OK] Log saved: D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD\USED\marsbar_conduct\_copy_log.xlsx


In [ ]:
## Final merged dF

In [63]:
# Paths
base_path = os.path.join(files, "finLIST-1120_8641_ALPS+TiV.xlsx")          # base (has 'PDI')
src1_path = os.path.join(files, "Auto_ALPS_DTI+CTI_1120.xlsx")                 # [1]
src2_path = os.path.join(files, "BrainVolumeStats_merged_1120.xlsx")              # [2]

# Load
base = pd.read_excel(base_path)
s1   = pd.read_excel(src1_path)
s2   = pd.read_excel(src2_path)
s2.columns = [str(c).strip() for c in s2.columns]
if "EstimatedTotalIntraCranialVol" in s2.columns:
    s2 = s2.drop(columns=["EstimatedTotalIntraCranialVol"])
cc_cols = ["CC_Posterior","CC_Mid_Posterior","CC_Central","CC_Mid_Anterior","CC_Anterior"]
for c in cc_cols:
    if c in s2.columns:
        s2[c] = pd.to_numeric(s2[c], errors="coerce")
s2["CorpusCallosum"] = s2[cc_cols].sum(axis=1, min_count=1)
rename_map = {
    "Left-Thalamus": "LT_Thalamus",
    "Left-Hippocampus": "LT_Hippocampus",
    "Left-Amygdala": "LT_Amygdala",
    "Left-choroid-plexus": "LT_ChoroidPlexus",
    "Right-Thalamus": "RT_Thalamus",
    "Right-Hippocampus": "RT_Hippocampus",
    "Right-Amygdala": "RT_Amygdala",
    "Right-choroid-plexus": "RT_ChoroidPlexus",
    # CC_* 개별도 유지(아래 최종 선택에서 사용)
    "CC_Posterior": "Pos_CC",
    "CC_Mid_Posterior": "MidPos_CC",
    "CC_Central": "Central_CC",
    "CC_Mid_Anterior": "MidAnt_CC",
    "CC_Anterior": "Ant_CC",
    "ctx-lh-middletemporal": "LT_MiddleTemporal_CTX",
    "ctx-lh-parahippocampal": "LT_ParaHippocampal_CTX",
    "ctx-lh-precuneus": "LT_Precuneus_CTX",
    "ctx-lh-insula": "LT_Insula_CTX",
    "ctx-rh-middletemporal": "RT_MiddleTemporal_CTX",
    "ctx-rh-parahippocampal": "RT_ParaHippocampal_CTX",
    "ctx-rh-precuneus": "RT_Precuneus_CTX",
    "ctx-rh-insula": "RT_Insula_CTX",
    "Left-Lateral-Ventricle": "LT_Lat_Ventricle",
    "CSF": "CSF",  # 그대로 유지
    "Right-Lateral-Ventricle": "RT_Lat_Ventricle",
    "ctx-lh-entorhinal": "LT_Entorhinal_CTX",
    "ctx-rh-entorhinal": "RT_Entorhinal_CTX",
}
s2 = s2.rename(columns=rename_map)
keep_order = [
    "LT_Thalamus","LT_Hippocampus","LT_Amygdala","LT_ChoroidPlexus",
    "RT_Thalamus","RT_Hippocampus","RT_Amygdala","RT_ChoroidPlexus",
    "CorpusCallosum",
    "Pos_CC","MidPos_CC","Central_CC","MidAnt_CC","Ant_CC",
    "LT_MiddleTemporal_CTX","LT_ParaHippocampal_CTX","LT_Precuneus_CTX","LT_Insula_CTX",
    "RT_MiddleTemporal_CTX","RT_ParaHippocampal_CTX","RT_Precuneus_CTX","RT_Insula_CTX",
    "LT_Lat_Ventricle","CSF","RT_Lat_Ventricle",
    "LT_Entorhinal_CTX","RT_Entorhinal_CTX",
]



# Check key
for df in (base, s1, s2):
    if "PDI" not in df.columns: raise KeyError("'PDI' column missing.")
    df["PDI"] = df["PDI"].astype(str).str.strip()

# Index base by PDI
base = base.set_index("PDI")

def update_overlap(base_df: pd.DataFrame, src_df: pd.DataFrame):
    src_df = src_df.copy().set_index("PDI")
    common = list(set(base_df.columns) & set(src_df.columns))
    if common:
        base_df.update(src_df[common])

# Update only overlapping columns (by PDI)
update_overlap(base, s1)
update_overlap(base, s2)


base["PID"] = base["PID"].astype(str).str.zfill(8)
base

,ADD/FAIL,PID,DDX,Group,Age,Sex_01,Sex,Education (Year),MMSE,CDR,...,LT_CTI_Assoc_Dyy,LT_CTI_Assoc_Dzz,LT_CTI-ALPS index,RT_CTI_Proj_Dxx,RT_CTI_Proj_Dyy,RT_CTI_Proj_Dzz,RT_CTI_Assoc_Dxx,RT_CTI_Assoc_Dyy,RT_CTI_Assoc_Dzz,RT_CTI-ALPS index
PDI,,,,,,,,,,,,,,,,,,,,,
EPT2022AD01_KDKM_NODDI32,NaN,00310282,AD,2,66,1,F,NaN,23,0.5,...,0.164354,0.097310,1.478380,0.054003,0.054375,0.240860,0.184253,0.287577,0.178141,1.024686
EPT2022AD02_SKM_NODDI32,NaN,00454439,AD,2,74,0,M,NaN,13,2.0,...,0.662543,0.608131,1.031772,0.571705,0.575767,0.810732,0.635722,0.745992,0.640786,0.992498
EPT2022AD03_KKS_NODDI32,NaN,00461257,SaMCI,1,74,1,F,12.0,27,0.5,...,0.218842,0.130708,1.237209,0.162839,0.164997,0.514460,0.213250,0.368805,0.217572,0.983061
EPT2022AD05_KIJD_NODDI32,NaN,00612238,AD,2,83,1,F,NaN,24,1.0,...,0.302673,0.241407,1.154961,0.320157,0.311376,0.667341,0.353844,0.461456,0.331211,1.048886
EPT2022AD063_BPB_NODDI32,NaN,00710431,SaMCI,1,74,1,F,6.0,26,0.5,...,0.279020,0.148243,1.073432,0.070293,0.067808,0.255883,0.215472,0.385115,0.222588,0.984049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
MREPT2020AD18_YHS_NODDI32,ADD,04058635,AD,2,86,1,F,1.0,14,1.0,...,0.432221,0.282059,1.092207,0.198349,0.185045,0.486609,0.217757,0.325501,0.195334,1.093927
MREPT2020AD24_YYS_NODDI32,ADD,08700631,AD,2,60,1,F,6.0,25,1.0,...,0.371340,0.217176,1.203538,0.093545,0.113938,0.350324,0.220411,0.385111,0.261687,0.835825
MREPT2020AD28_YYL_NODDI32,ADD,05533209,AD,2,80,1,F,NaN,13,1.0,...,0.585913,0.396878,1.075353,0.263975,0.261592,0.578273,0.334748,0.527785,0.340203,0.994895


In [61]:
# Save
out_path  = os.path.join(files, "1120_8641_ALPS+TiV_UPDATED.xlsx")

base.reset_index().to_excel(out_path, index=False)
print(f"[OK] Saved: {out_path}")

[OK] Saved: D:\JHAhn\files\1120_8641_ALPS+TiV_UPDATED.xlsx
